RAMAD public-package overview

This notebook reads the public QA corpus from the companion Zenodo package, previews the training-input structure, and lists the executable entry points. It uses only the Python standard library and does not call an external model service.


In [ ]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next((path for path in candidates if (path / "benchmark" / "run_benchmark.py").is_file() and (path / "src" / "ramad_model").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or its notebooks directory.")

COMPANION_DIR = REPO_ROOT.parent / "ramad_zenodo_package"
CORPUS_PATH = COMPANION_DIR / "training" / "RAMAD_domain_QA_1000.jsonl"
if not CORPUS_PATH.is_file():
    raise FileNotFoundError("Companion package not found: download the Zenodo archive (DOI: 10.5281/zenodo.17809450) and extract it as 'ramad_zenodo_package' beside this repository.")

ENTRY_POINTS = [
    REPO_ROOT / "src" / "ramad_model" / "train_sft_lora.py",
    REPO_ROOT / "src" / "ramad_model" / "run_inference.py",
    REPO_ROOT / "src" / "ramad_rag" / "build_index.py",
    REPO_ROOT / "src" / "ramad_rag" / "rag_qa.py",
    REPO_ROOT / "benchmark" / "run_benchmark.py",
    REPO_ROOT / "benchmark" / "aggregate_scores.py",
    REPO_ROOT / "benchmark" / "frontier" / "run_frontier_eval.py",
    REPO_ROOT / "interactive_scoring.py",
]
print(REPO_ROOT)
print(COMPANION_DIR)


In [ ]:
def load_jsonl(path):
    with path.open("r", encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]

qa_records = load_jsonl(CORPUS_PATH)
required_fields = {"id", "instruction", "input", "output", "topic", "language"}
assert len(qa_records) == 1000
assert all(required_fields == set(record) for record in qa_records)
print(len(qa_records))
print(dict(sorted(Counter(record["topic"] for record in qa_records).items())))
print(dict(sorted(Counter(record["language"] for record in qa_records).items())))


In [ ]:
record = qa_records[0]
parts = [f"Topic: {record['topic']}", f"Language: {record['language']}", "", f"Instruction:\n{record['instruction']}"]
if record["input"].strip():
    parts.extend(["", f"Input:\n{record['input']}"])
print("\n".join(parts))
print("\nTarget output:\n" + record["output"])
print(REPO_ROOT / "src" / "ramad_model" / "train_sft_lora.py")


In [ ]:
for path in ENTRY_POINTS:
    print(path.relative_to(REPO_ROOT), path.is_file())


In [ ]:
commands = [[sys.executable, str(path), "--help"] for path in ENTRY_POINTS]
for command in commands:
    print(" ".join(command))


Entry points

`src/ramad_model/` contains the training and inference scripts.

`src/ramad_rag/` contains the index-building and question-answering scripts.

`benchmark/` contains the common-question scoring workflow; `benchmark/frontier/` contains the frontier-model comparison runner.

`interactive_scoring.py` records and summarizes human expert scores.

`run_public_package.ps1` checks the corpus and command-line availability.
